1. Treinamento Final (3 sementes)

In [ ]:
import sys
from pathlib import Path
import importlib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import src.train
import src.models
importlib.reload(src.train)
importlib.reload(src.models)

from src.train import set_seed, prepare_dataset, treinar_modelo
from src.models import MLP

RANDOM_STATE = 42
SEEDS_FINAIS = [42, 100, 202]
BATCH_SIZE = 256

set_seed(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Rodando em:", device)

MODELS_DIR = ROOT_DIR / "models"
FIG_DIR = ROOT_DIR / "reports" / "figures"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

reg_data = prepare_dataset(
    csv_path=str(ROOT_DIR / "data" / "processed" / "dataset_regression.csv"),
    task="regression",
    target_col="TARGET",
    random_state=RANDOM_STATE,
    batch_size=BATCH_SIZE,
)

clf_data = prepare_dataset(
    csv_path=str(ROOT_DIR / "data" / "processed" / "dataset_classification.csv"),
    task="classification",
    target_col="TARGET",
    random_state=RANDOM_STATE,
    batch_size=BATCH_SIZE,
)


def treinar_e_avaliar_final(
    task: str,
    data_dict: dict,
    hidden_units: tuple,
    activation: str,
    lr: float,
    dropout: float,
    nome_arquivo_pt: str,
    n_epocas: int = 80,
    paciencia: int = 15,
):
    loss_fn = torch.nn.MSELoss() if task == "regression" else torch.nn.BCEWithLogitsLoss()
    melhor_val_global = float("inf")
    melhor_modelo = None
    melhor_seed = None
    historico_melhor = None
    metricas_seeds = []

    for s in SEEDS_FINAIS:
        set_seed(s)
        modelo = MLP(
            in_features=data_dict["n_features"],
            out_features=1,
            hidden_units=hidden_units,
            activation=activation,
            dropout=dropout,
        ).to(device)

        otim = torch.optim.Adam(modelo.parameters(), lr=lr)
        res = treinar_modelo(
            modelo=modelo,
            dl_treino=data_dict["loaders"]["train"],
            dl_val=data_dict["loaders"]["val"],
            loss_fn=loss_fn,
            otimizador=otim,
            device=device,
            n_epocas=n_epocas,
            paciencia=paciencia,
        )

        modelo.eval()
        preds_test_list = []
        with torch.no_grad():
            for xb, _ in data_dict["loaders"]["test"]:
                xb = xb.to(device)
                out = modelo(xb)
                if task == "regression":
                    preds_test_list.append(out.cpu().numpy().ravel())
                else:
                    preds_test_list.append(torch.sigmoid(out).cpu().numpy().ravel())

        y_test_raw = data_dict["raw_splits"]["y_test"]
        preds_arr = np.concatenate(preds_test_list)

        if task == "regression":
            y_test_scaled = data_dict["scaled_splits"]["y_test"]
            scaler_y = None
            scalers_dict = data_dict.get("scalers")
            if isinstance(scalers_dict, dict):
                for k in ["y", "target", "TARGET"]:
                    if k in scalers_dict and scalers_dict[k] is not None:
                        scaler_y = scalers_dict[k]
                        break

            if scaler_y is not None:
                y_test_pred_real = scaler_y.inverse_transform(preds_arr.reshape(-1, 1)).ravel()
                y_test_alvo = y_test_raw
            else:
                y_test_pred_real = preds_arr
                y_test_alvo = y_test_raw

            rmse = float(np.sqrt(mean_squared_error(y_test_alvo, y_test_pred_real)))
            mae = float(mean_absolute_error(y_test_alvo, y_test_pred_real))
            r2 = float(r2_score(y_test_alvo, y_test_pred_real))
            rmse_scaled = float(np.sqrt(mean_squared_error(y_test_scaled, preds_arr)))

            metricas_seeds.append({
                "seed": s,
                "val_loss": res["melhor_val"],
                "rmse_scaled": rmse_scaled,
                "rmse": rmse,
                "mae": mae,
                "r2": r2,
            })
        else:
            y_true_int = np.asarray(y_test_raw).ravel().astype(int)
            y_pred_int = (preds_arr >= 0.5).astype(int)
            metricas_seeds.append({
                "seed": s,
                "val_loss": res["melhor_val"],
                "roc_auc": float(roc_auc_score(y_true_int, preds_arr)),
                "pr_auc": float(average_precision_score(y_true_int, preds_arr)),
                "f1_05": float(f1_score(y_true_int, y_pred_int, zero_division=0)),
                "precision_05": float(precision_score(y_true_int, y_pred_int, zero_division=0)),
                "recall_05": float(recall_score(y_true_int, y_pred_int, zero_division=0)),
                "acc_05": float(accuracy_score(y_true_int, y_pred_int)),
            })

        if res["melhor_val"] < melhor_val_global:
            melhor_val_global = res["melhor_val"]
            melhor_modelo = modelo
            melhor_seed = s
            historico_melhor = res["historico"]

    caminho_pt = MODELS_DIR / nome_arquivo_pt
    torch.save(
        {
            "task": task,
            "seed": melhor_seed,
            "val_loss": melhor_val_global,
            "model_config": {
                "in_features": data_dict["n_features"],
                "out_features": 1,
                "hidden_units": hidden_units,
                "activation": activation,
                "dropout": dropout,
            },
            "state_dict": melhor_modelo.state_dict(),
        },
        caminho_pt,
    )
    print(f"\n[{task.upper()}] Salvo em: {caminho_pt} (melhor seed={melhor_seed}, val_loss={melhor_val_global:.6f})")
    df_met = pd.DataFrame(metricas_seeds)
    print(df_met.to_string(index=False))
    print("Médias no Teste:")
    print(df_met.drop(columns=["seed"], errors="ignore").mean().round(4).to_dict())

    return melhor_modelo, df_met, historico_melhor


modelo_clf_final, met_clf_test, hist_clf_final = treinar_e_avaliar_final(
    task="classification",
    data_dict=clf_data,
    hidden_units=(64, 32),
    activation="ReLU",
    lr=1e-3,
    dropout=0.1,
    nome_arquivo_pt="mlp_final_classificacao.pt",
)

Rodando em: cpu

[REGRESSION] Salvo em: /home/ariadnydantas/Área de trabalho/IA/Relatorio01_IA/models/mlp_final_regressao.pt (melhor seed=42, val_loss=0.001174)
 seed  val_loss  rmse_scaled     rmse      mae       r2
   42  0.001174     0.036293 0.036303 0.012673 0.998671
  100  0.001210     0.036349 0.036474 0.014651 0.998659
  202  0.001190     0.036402 0.036349 0.013009 0.998668
Médias no Teste:
{'val_loss': 0.0012, 'rmse_scaled': 0.0363, 'rmse': 0.0364, 'mae': 0.0134, 'r2': 0.9987}

[CLASSIFICATION] Salvo em: /home/ariadnydantas/Área de trabalho/IA/Relatorio01_IA/models/mlp_final_classificacao.pt (melhor seed=100, val_loss=0.049915)
Empty DataFrame
Columns: []
Index: []
Médias no Teste:


KeyError: "['seed'] not found in axis"

In [3]:
ckpt_reg = torch.load(MODELS_DIR / "mlp_final_regressao.pt", map_location=device)
mod_reg_chk = MLP(**ckpt_reg["model_config"]).to(device)
mod_reg_chk.load_state_dict(ckpt_reg["state_dict"])
mod_reg_chk.eval()

preds_reg_t = []
with torch.no_grad():
    for xb, _ in reg_data["loaders"]["test"]:
        preds_reg_t.append(mod_reg_chk(xb.to(device)).cpu().numpy().ravel())
y_pred_reg_t = np.concatenate(preds_reg_t)
y_true_reg_t = np.asarray(reg_data["raw_splits"]["y_test"]).ravel()

print("=== TESTE FINAL — REGRESSÃO (checkpoint .pt) ===")
print(f"RMSE : {np.sqrt(mean_squared_error(y_true_reg_t, y_pred_reg_t)):.4f}")
print(f"MAE  : {mean_absolute_error(y_true_reg_t, y_pred_reg_t):.4f}")
print(f"R²   : {r2_score(y_true_reg_t, y_pred_reg_t):.4f}")

ckpt_clf = torch.load(MODELS_DIR / "mlp_final_classificacao.pt", map_location=device)
mod_clf_chk = MLP(**ckpt_clf["model_config"]).to(device)
mod_clf_chk.load_state_dict(ckpt_clf["state_dict"])
mod_clf_chk.eval()

probs_clf_t = []
with torch.no_grad():
    for xb, _ in clf_data["loaders"]["test"]:
        probs_clf_t.append(torch.sigmoid(mod_clf_chk(xb.to(device))).cpu().numpy().ravel())
y_prob_clf_t = np.concatenate(probs_clf_t)
y_true_clf_t = np.asarray(clf_data["raw_splits"]["y_test"]).ravel().astype(int)
y_pred_clf_t = (y_prob_clf_t >= 0.5).astype(int)

print("\n=== TESTE FINAL — CLASSIFICAÇÃO (checkpoint .pt, limiar=0.50) ===")
print(f"Accuracy  : {accuracy_score(y_true_clf_t, y_pred_clf_t):.4f}")
print(f"Precision : {precision_score(y_true_clf_t, y_pred_clf_t, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_true_clf_t, y_pred_clf_t, zero_division=0):.4f}")
print(f"F1-score  : {f1_score(y_true_clf_t, y_pred_clf_t, zero_division=0):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_true_clf_t, y_prob_clf_t):.4f}")
print(f"PR-AUC    : {average_precision_score(y_true_clf_t, y_prob_clf_t):.4f}")
print("Matriz de confusão:\n", confusion_matrix(y_true_clf_t, y_pred_clf_t))

=== TESTE FINAL — REGRESSÃO (checkpoint .pt) ===
RMSE : 0.0363
MAE  : 0.0127
R²   : 0.9987

=== TESTE FINAL — CLASSIFICAÇÃO (checkpoint .pt, limiar=0.50) ===
Accuracy  : 0.9854
Precision : 1.0000
Recall    : 0.9274
F1-score  : 0.9623
ROC-AUC   : 0.9985
PR-AUC    : 0.9952
Matriz de confusão:
 [[39963     0]
 [  729  9308]]
